## 1. Drop-Medallion

Revoca los permisos otorgados por `seguridad/5.Grants_Medallion` y elimina tablas y archivos Delta del proyecto de futbol.

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
dbutils.widgets.text("storageName", "s3-football-bucket")
dbutils.widgets.text("catalogo", "football_dev")
dbutils.widgets.text("usuarios", "user1@gmail.com, user2@gmail.com, user3@gmail.com")

In [0]:
%python
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")
usuarios = [u.strip() for u in dbutils.widgets.get("usuarios").split(",") if u.strip()]

### Revocacion de permisos

Se ejecuta antes de borrar: un `REVOKE` sobre una tabla o catalogo que ya no existe falla.

In [0]:
%python
TABLAS = {
    "bronze": ["leagues", "matches", "clubs"],
    "silver": ["matches_transformed"],
    "gold": ["season_stats", "standings", "calendar_stats", "club_elo", "match_predictions"],
}

if spark.sql(f"SHOW CATALOGS LIKE '{catalogo}'").count() == 0:
    print(f"El catalogo {catalogo} no existe: no hay permisos que revocar")
else:
    for usuario in usuarios:
        for esquema, tablas in TABLAS.items():
            for tabla in tablas:
                nombre = f"{catalogo}.{esquema}.{tabla}"
                if spark.catalog.tableExists(nombre):
                    spark.sql(f"REVOKE SELECT ON TABLE {nombre} FROM `{usuario}`")
            if spark.catalog.databaseExists(f"{catalogo}.{esquema}"):
                spark.sql(f"REVOKE USE SCHEMA ON SCHEMA {catalogo}.{esquema} FROM `{usuario}`")
        spark.sql(f"REVOKE USE CATALOG ON CATALOG {catalogo} FROM `{usuario}`")
        print(f"Permisos revocados a {usuario}")

### Eliminacion tablas Bronze


In [0]:
%sql
DROP TABLE IF EXISTS ${catalogo}.bronze.leagues;
DROP TABLE IF EXISTS ${catalogo}.bronze.matches;
DROP TABLE IF EXISTS ${catalogo}.bronze.clubs;

In [0]:
%python
dbutils.fs.rm(f"s3://{storageName}/{catalogo}/bronze/leagues", True)
dbutils.fs.rm(f"s3://{storageName}/{catalogo}/bronze/matches", True)
dbutils.fs.rm(f"s3://{storageName}/{catalogo}/bronze/clubs", True)

### Eliminacion tablas Silver


In [0]:
%sql
DROP TABLE IF EXISTS ${catalogo}.silver.matches_transformed;

In [0]:
%python
dbutils.fs.rm(f"s3://{storageName}/{catalogo}/silver/matches_transformed", True)

### Eliminacion tablas Golden


In [0]:
%sql
DROP TABLE IF EXISTS ${catalogo}.gold.season_stats;

In [0]:
%python
dbutils.fs.rm(f"s3://{storageName}/{catalogo}/gold/season_stats", True)

### Eliminacion de catalogo


In [0]:
%sql
DROP CATALOG IF EXISTS ${catalogo} CASCADE;